# 5  Groups of patients by service use

**Goal:** find groups of patients without any labels, choose the number of groups, see them in one picture and give each group a name.

**Data:** 3,000 synthetic patients and how they used health services in the last 12 months.

**The data used here:**
- `patients_service_use.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/patients_service_use.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/patients_service_use.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

## Step 1  Patients without labels

In [ ]:
#@title Step 1  Load the patients
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
df = pd.read_csv(DATA + "patients_service_use.csv")
F = ["gp_visits_12m", "specialist_visits_12m", "emergency_visits_12m", "prescriptions_12m", "missed_appointments_12m", "telehealth_sessions_12m", "lab_tests_12m", "chronic_conditions", "distance_to_clinic_km"]
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df["gp_visits_12m"] + df["specialist_visits_12m"] + np.random.default_rng(0).random(len(df)) * .6, df["prescriptions_12m"] + np.random.default_rng(1).random(len(df)) * .6, s=8, color="#999999")
ax.set_xlabel("doctor visits per year"); ax.set_ylabel("prescriptions per year"); ax.set_title("3,000 patients, no labels"); plt.show()
box("What you see", "There is no target here. We want the data to show which <b>groups</b> of patients exist, so services can be planned for each group.", "info")

## Step 2  Scale the inputs first

In [ ]:
#@title Step 2  Should we scale?  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "No, the raw numbers are fine", "Yes, so each input counts equally"]
display(df[F].describe().loc[["mean", "std", "max"]].round(1))
_ = answer(question, {
    "No, the raw numbers are fine": (False, "Distance to the clinic goes up to 40 km, emergency visits rarely above 5. Without scaling, distance would decide the groups almost alone."),
    "Yes, so each input counts equally": (True, "Scaling puts every input on the same footing: each is measured in standard deviations from its average."),
})
Z = StandardScaler().fit_transform(df[F])

## Step 3  Find the groups

In [ ]:
#@title Step 3  Move the slider: how many groups?  (then move the slider through all values)
P = PCA(n_components=2).fit_transform(Z)
SIL = {}
def show(k):
    lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Z)
    if k not in SIL: SIL[k] = silhouette_score(Z, lab, sample_size=1500, random_state=42)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2, 1]})
    ax[0].scatter(P[:, 0], P[:, 1], c=lab, cmap="tab10", s=7, alpha=.7); ax[0].set_title(f"k-means with {k} groups (two-dimensional view)")
    ax[0].set_xticks([]); ax[0].set_yticks([])
    ks = range(2, 8); ax[1].bar([str(x) for x in ks], [SIL.get(x, 0) for x in ks], color=[RED if x == k else INK for x in ks]); ax[1].set_ylim(0, 0.5)
    ax[1].set_title("silhouette score (higher = clearer groups)"); ax[1].set_xlabel("k"); plt.show()
_ = live(show, k=islider("Number of groups k", 2, 7, 2))

In [ ]:
#@title Step 3b  Your choice of k  (choose k on the right, then click &#9654;)
k = 4 #@param {type:"slider", min:2, max:7, step:1}
km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z); df["group"] = km.labels_
if k in (2, 3): box("Possible", "The score is high, but the groups are broad: very different patients end up together. Look at step 4 and check whether they make sense.", "try")
elif k == 4: box("A good choice", "The score is close to the best, and the four groups are clearly different in how patients use services.", "ok")
else: box("Possible", "More groups, but lower scores: some groups split patients who behave alike. Fewer groups are easier to plan for.", "try")

## Step 4  PCA: all inputs in one picture

In [ ]:
#@title Step 4  Show the groups
pca = PCA(n_components=2).fit(Z); P = pca.transform(Z)
fig, ax = plt.subplots(figsize=(9, 6))
for g in sorted(df["group"].unique()):
    m = df["group"] == g; ax.scatter(P[m, 0], P[m, 1], s=8, alpha=.7, label=f"group {g + 1}")
ax.legend(markerscale=3); ax.set_xlabel("new input 1 (combination of nine)"); ax.set_ylabel("new input 2"); ax.set_xticks([]); ax.set_yticks([]); plt.show()
box("What you see", f"PCA combines the nine inputs into two new ones that keep {pca.explained_variance_ratio_.sum():.0%} of the variation, so the groups fit in one picture. "
    "The new axes are hard to interpret: describe the groups with the original inputs instead, in the next step.", "info")

## Step 5  Describe and name the groups

In [ ]:
#@title Step 5  What makes each group different?
prof = df.groupby("group")[F + ["age"]].mean()
rel = (prof - df[F + ["age"]].mean()) / df[F + ["age"]].std()
fig, ax = plt.subplots(figsize=(12, 0.7 * len(prof) + 1.5))
im = ax.imshow(rel.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(rel.columns))); ax.set_xticklabels([c.replace("_12m", "").replace("_", " ") for c in rel.columns], rotation=35, ha="right")
ax.set_yticks(range(len(rel))); ax.set_yticklabels([f"group {g + 1} ({(df['group'] == g).sum()})" for g in rel.index])
for i in range(rel.shape[0]):
    for j in range(rel.shape[1]): ax.text(j, i, f"{prof.values[i, j]:.1f}", ha="center", va="center", fontsize=10)
ax.set_title("Average per group (red: above average, blue: below)"); plt.show()

In [ ]:
#@title Step 5b  Give each group a name  (type names on the right, then click &#9654;)
group_1 = "" #@param {type:"string"}
group_2 = "" #@param {type:"string"}
group_3 = "" #@param {type:"string"}
group_4 = "" #@param {type:"string"}
names = [group_1, group_2, group_3, group_4]
for g in sorted(df["group"].unique())[:4]:
    p = prof.loc[g]; top = (rel.loc[g, F]).sort_values(ascending=False).index[:2]
    nm = names[g] or "(no name yet)"
    box(f"Group {g + 1}: {nm}", f"{(df['group'] == g).sum()} patients. Stands out by: {', '.join(t.replace('_12m', '').replace('_', ' ') for t in top)}.", "step")
if not all(names[:min(4, df['group'].nunique())]): box("Your turn", "Type a short name for each group in the fields on the right, then click &#9654; again.", "try")

## Result

In [ ]:
#@title Result
box("Do you see it now?", "Without any labels, the data shows distinct groups of patients: for example people who come rarely, people with chronic conditions, "
    "frequent emergency visitors and people who prefer telehealth. <b>Now the service can plan for each group differently.</b> "
    "People who know the service give the names and check that the groups make sense.", "see")

---
## Stretch (optional, for those who code)

In [ ]:
# Stretch: try hierarchical clustering on 500 patients and compare with k-means
from sklearn.cluster import AgglomerativeClustering
s = np.random.default_rng(1).choice(len(Z), 500, replace=False)
pd.crosstab(df["group"].values[s], AgglomerativeClustering(n_clusters=k).fit_predict(Z[s]))